## 1. Install Required libraries

In [ ]:
!pip install langchain langchain-community langchain-ollama faiss-cpu pymupdf

# 2. Setup ollama
Install Ollama from:  
https://ollama.com  
Then pull models  
-    ollama pull llama3.2  
-    ollama pull nomic-embed-text  

# 3. Import libraries

In [21]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

# 4. Load Document
we will load a PDF document from the data folder

In [22]:
loader = PyMuPDFLoader("../Gen_ai_data/sample_ai_document.pdf")

docs = loader.load()

len(docs)

1

# 5. Split Documents into chunks
Large documents must be split before creating embeddings

In [23]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(docs)

len(chunks)

2

# 6. Create Embeddings
Embeddings convert into numerical text

In [24]:
embeddings = OllamaEmbeddings(
    model = "nomic-embed-text"
)

# 7. Create Vector Database
we store embeddings inside Faiss

In [25]:
vectorstore= FAISS.from_documents(
    chunks, 
    embeddings
)

# 8. Create Retriever 
The retriever finds relevant document chunks.


In [26]:
retriever = vectorstore.as_retriever(
    search_kwargs = {"k":3}
)

# 9. Create LLM
we use Ollama's local LLM

In [27]:
llm = ChatOllama(
    model = "llama3.2"
)

# 10. Add Conversation Memory
This allows the chatbot to remember previous messages.

In [28]:
memory = ConversationBufferMemory(
    memory_key = "chat_history",
    return_messages = True
)

# 11. Create Conversation Rag Chain

In [29]:
qa_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever = retriever,
    memory = memory
)

# 12. Ask Questions

In [30]:
query = "What is Artificial Intelligence?"
result = qa_chain.invoke({"question":query})
result["answer"]

'Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn. It enables systems to perform tasks such as problem solving, speech recognition, decision making, and language translation.'

# 13. Simple Chat Loop
You can turn this into an interactive chatbot

In [31]:
while True:
    question = input("You: ")

    if question.lower() == 'exit':
        break

    result = qa_chain.invoke({"question":question})

    print("AI:", result["answer"])

You:  hii, my name is aditya


AI: I don't know the answer to that question.


You:  what is AI?


AI: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn.


You:  exit


# 14.

# 15.